In [29]:
import pandas as pd
import numpy as np
import io
from tqdm import tqdm
from curl_cffi import requests as cureq
import cloudscraper
from IPython.display import clear_output

In [30]:
pd.set_option('display.max_columns', None)

In [31]:
def list_files(dataset_id):
    url = f"https://ckan.pbh.gov.br/api/3/action/package_show?id={dataset_id}"
    # scraper = cloudscraper.create_scraper()  # burlar o Cloudflare
    # response = scraper.get(url)
    response = cureq.get(url, impersonate = "chrome")

    if response.status_code == 200:
        data = response.json()
        resources = data['result']['resources']

        obj_files = []
        for resource in resources:
            df_file = pd.DataFrame(
                data = {"name": resource["name"], "url": resource["url"], "id": resource["id"], "format": resource["format"]},
                index = [0])
            obj_files.append(df_file)

        df_files = pd.concat(objs = obj_files, ignore_index = True).reset_index(drop = True)
        return df_files
    else:
        raise Exception(f"API ERROR: {response.status_code} - {response.text}")

def get_csv_file(url, dataset = None):
    if url == "":
        return None

    # headers = {
    #     "User-Agent": "Mozilla/5.0",
    #     "Referer": "https://ckan.pbh.gov.br"
    # }

    # scraper = cloudscraper.create_scraper()
    # response = scraper.get(url, headers = headers)
    response = cureq.get(url, impersonate = "chrome")

    if response.status_code == 200:
        # Get the binary content
        csv_data = io.BytesIO(response.content)

        # read the csv file using pandas
        df = pd.DataFrame()
        try:
            df = pd.read_csv(csv_data, sep = ';', encoding = 'utf-8')
            print(f"{url}: successful downloaded data!")
        except Exception as e:
            # Try another encoding if the first fail
            csv_data.seek(0)
            df = pd.read_csv(csv_data, sep = ';', encoding = 'latin-1')
            print("successful downloaded data (UTF-8)!")
    else:
        print(f"ERROR TO DOWNLOAD {url}: {response.status_code}")
    df.columns = [item.replace("çã", "ca").replace("ï»¿", "").replace("(", "").replace(")", "") for item in df.columns]
    df.columns = df.columns.str.strip()
    df.columns = df.columns.str.lower()
    df.columns = [item.replace(" ", "_") for item in df.columns]
    df.columns = [item.replace("_de_", "_") for item in df.columns]
    df.columns = [item.replace('"', "") for item in df.columns]

    if dataset == "itbi":
        df = df.rename(columns = {"zona_uso": "zona_uso_itbi", "data_quitacao": "data_quitacao_transacao"})

    df = df.assign(urlfile = url)

    clear_output(wait = True)
    return df



In [32]:
dict_datasets_id = {"itbi": "0e13bf71-5355-47ce-8607-966413b08c0a",
                    "enderecamento": "394da7d4-3e74-4a2d-8fb6-6cb4d65d3451",
                    "cadastro_imobiliario_reg_pampulha": "e4f8fc31-df75-4f0b-9aab-f1abe1512d3f",
                    "cadastro_imobiliario_reg_oeste": "da432bf7-dcd4-463d-b10a-ab4922c8e2b3",
                    "cadastro_imobiliario_reg_norte": "df1cea37-92d0-4d52-917a-b1ed10bae4db",
                    "cadastro_imobiliario_reg_noroeste": "afb79a2b-69b3-4482-82d5-df3025070fd0",
                    "cadastro_imobiliario_reg_nordeste": "f7f03edc-39bc-4e17-aa94-a52d8bf56f04",
                    "cadastro_imobiliario_reg_barreiro": "41c80bc4-e0e3-4d06-9547-a8a1051c9fb6",
                    "cadastro_imobiliario_reg_leste": "f0745d2c-8b5c-4024-afb2-886c5de148e0",
                    "cadastro_imobiliario_reg_hipercentro": "4000eb81-51fc-47d7-931e-445099e0f45a",
                    "cadastro_imobiliario_reg_centrosul": "1ab07865-1a30-4e09-88c1-17d2183b2ea0",
                    "cadastro_imobiliario_reg_vendanova": "3ddf7e20-812d-4e74-99cd-814c942cb8c3",
                    "atividades_economicas_old": "3449df83-944b-4672-835d-d3e4a7bf7f48",
                    "atividades_economicas": "33e9dcb1-126f-4cde-8c80-d1927a965430",
                    "atividades_economicas_autonomos": "0e6890a3-6957-4d54-b6a6-0c77b97b1b88",
                    "edificacoes_licenciadas": "602b0331-286a-4a59-a987-6f6f38f6ebec",
                    "qtd_lancamentos_iptu_bairro": "177e1466-187d-47df-8638-1d02a060afc6",
                    "baixa_construcoes": "aa138ff1-7229-4101-ba1c-399eee7de8be"}

dict_tipo_construtivo = {"AP": "APARTAMENTO", "CA": "CASA", "SL": "SALA", "LV": "LOTE VAGO", "VC": "VAGA DE GARAGEM NAO RESIDENCIAL",
                         "LJ": "LOJA", "VR": "VAGA DE GARAGEM RESIDENCIAL", "BA": "BARRACAO", "GP": "GALPAO",
                         "AC": "APARTAMENTO COM OCUPACAO NAO RESIDENCIAL", "CC": "CASA COM OCUPACAO NAO RESIDENCIAL",
                         "BC": "BARRACAO COM OCUPACAO NAO RESIDENCIAL", "VV": "VAGA DE GARAGEM NAO RESIDENCIAL"}



In [33]:
list_resources = []
for key, value in tqdm(dict_datasets_id.items()):
    print(f"\n{key}")
    df_resource = list_files(value)\
    .assign(dataset = key)

    list_resources.append(df_resource)
    clear_output(wait = True)

df_resources = pd.concat(objs = list_resources, ignore_index = True)

100%|██████████| 18/18 [00:17<00:00,  1.00it/s]


In [34]:
df_itbi = df_resources\
.query("dataset == 'itbi' & format == 'CSV'")\
.apply(lambda df: get_csv_file(url = df["url"], dataset = "itbi"), axis = 1)

df_itbi = pd.concat(objs = list(df_itbi))\
.assign(area_terreno_total = lambda df: df["area_terreno_total"].str.replace(".", "").str.replace(",", ".").astype("float"))\
.assign(area_construida_adquirida = lambda df: df["area_construida_adquirida"].str.replace(".", "").str.replace(",", ".").astype("float"))\
.assign(area_adquirida_unidades_somadas = lambda df: df["area_adquirida_unidades_somadas"].str.replace(".", "").str.replace(",", ".").astype("float"))\
.assign(valor_declarado = lambda df: df["valor_declarado"].str.replace(".", "").str.replace(",", ".").astype("float"))\
.assign(valor_base_calculo = lambda df: df["valor_base_calculo"].str.replace(".", "").str.replace(",", ".").astype("float"))\
.assign(fracao_ideal_adquirida = lambda df: df["fracao_ideal_adquirida"].str.replace(",", ".").astype("float"))\
.assign(data_quitacao_transacao = lambda df: pd.to_datetime(df["data_quitacao_transacao"], format = "%d/%m/%Y"))\
.assign(ano_construcao_unidade = lambda df: [np.nan if i == 0 or i < 1800 and i > 2100 else i for i in df['ano_construcao_unidade']])\
.assign(tipo_construtivo_preponderante = lambda df: df["tipo_construtivo_preponderante"].map(dict_tipo_construtivo))\
.assign(urlfile = lambda df: df["urlfile"].str.split("/").apply(lambda x: x[-1]))

https://ckan.pbh.gov.br/dataset/0e13bf71-5355-47ce-8607-966413b08c0a/resource/9bd075e2-2abe-42d7-a1e2-b68a18245172/download/pda_itbi_relatorio_202511.csv: successful downloaded data!


In [35]:
df_itbi

,endereco,bairro,ano_construcao_unidade,area_terreno_total,area_construida_adquirida,area_adquirida_unidades_somadas,padrao_acabamento_unidade,fracao_ideal_adquirida,tipo_construtivo_preponderante,descricao_tipo_ocupacao_unidade,valor_declarado,valor_base_calculo,zona_uso_itbi,data_quitacao_transacao,urlfile
0,AVE AFONSO PENA 3924 - GARAGE 60 - CRUZEIRO - ...,CRUZEIRO,1976.0,1119.00,28.53,28.53,P3,0.004470,VAGA DE GARAGEM NAO RESIDENCIAL,NAO RESIDENCIAL,1000.00,11411.56,ZA,2008-01-02,pda_itbi_relatorio_200801_a_202405.csv
1,AVE AMAZONAS 718 - APT 704 - CENTRO - 30180-00...,CENTRO,1960.0,1030.00,126.99,126.99,P2,0.007197,APARTAMENTO,RESIDENCIAL,85000.00,85000.00,ZHIP,2008-01-02,pda_itbi_relatorio_200801_a_202405.csv
2,AVE AUGUSTO DE LIMA 1276 - APT 301 - BARRO PRE...,BARRO PRETO,1978.0,544.00,135.55,135.55,P3,0.025843,APARTAMENTO,RESIDENCIAL,121500.00,121500.00,ZCBH,2008-01-02,pda_itbi_relatorio_200801_a_202405.csv
3,AVE AUGUSTO DE LIMA 1276 - GARAGE 14 - BARRO P...,BARRO PRETO,1978.0,544.00,11.45,11.45,P3,0.002182,VAGA DE GARAGEM RESIDENCIAL,RESIDENCIAL,13500.00,13500.00,ZCBH,2008-01-02,pda_itbi_relatorio_200801_a_202405.csv
4,AVE AUGUSTO DE LIMA 233 - SALA 1439 - CENTRO -...,CENTRO,1967.0,4426.00,25.20,25.20,P3,0.000450,SALA,NAO RESIDENCIAL,9041.00,10354.49,ZHIP,2008-01-02,pda_itbi_relatorio_200801_a_202405.csv
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2233,RUA VARGINHA 463 - BLOCO A APT 1503 - COLEGIO ...,COLEGIO BATISTA,1983.0,1578.00,107.00,107.00,P3,0.012202,APARTAMENTO,RESIDENCIAL,219072.01,472512.00,ZAP,2025-11-28,pda_itbi_relatorio_202511.csv
2234,AVE BIAS FORTES 1577 - GARAGE 14 - BARRO PRETO...,BARRO PRETO,1983.0,375.00,12.00,12.00,P3,0.003043,VAGA DE GARAGEM RESIDENCIAL,RESIDENCIAL,26863.85,35078.40,ZCBH,2025-11-30,pda_itbi_relatorio_202511.csv
2235,RUA HENRIQUE GORCEIX 2120 - BLOCO V APT 403 - ...,JARDIM MONTANHES,1990.0,4755.29,70.70,70.70,P2,0.008929,APARTAMENTO,RESIDENCIAL,294000.00,294000.00,ZAR2,2025-11-30,pda_itbi_relatorio_202511.csv
2236,RUA MARIA HEILBUTH SURETTE 1312 - APT 1201 - B...,BURITIS,2025.0,3308.00,194.48,194.48,P4,0.017478,APARTAMENTO,RESIDENCIAL,1245016.12,1245016.12,ZAR2,2025-11-30,pda_itbi_relatorio_202511.csv


In [36]:
df_cadastro_imobiliario = df_resources\
.query("dataset.str.contains('cadastro_imobiliario') & format == 'CSV'")\
.groupby("dataset")\
.apply(lambda df: df.iloc[[-1]], include_groups = False)\
.reset_index()\
.apply(lambda df: get_csv_file(url = df["url"]), axis = 1)

df_cadastro_imobiliario = pd.concat(objs = list(df_cadastro_imobiliario), ignore_index = True)\
.assign(urlfile = lambda df: df["urlfile"].str.split("/").apply(lambda x: x[-1]))\
.drop(columns = ["frequencia_coleta", "ind_meio_fio", "ind_pavimentacao", "ind_arborizacao", "ind_galeria_pluvial",
                 "ind_iluminacao_publica", "ind_rede_esgoto", "ind_rede_agua", "ind_rede_telefonica"])\
.astype(dtype = {"cep": "str", "numero_imovel": "str", "nulotctm": "str"})\
.assign(numero_imovel = lambda df: df["numero_imovel"].str.replace("\\.0", "", regex = True))\
.assign(cep = lambda df: df["cep"].str.replace("^0$", "", regex = True))

KeyboardInterrupt: 

In [19]:
# df_resource = df_resources\
# .query("dataset == 'enderecamento' & format == 'CSV'")\
# .reset_index()\
# .sort_values("index", ascending = False)\
# .reset_index(drop = True)

# df_enderecamento = pd.DataFrame()
# for index in tqdm(range(df_resource.shape[0])):
#     df_ = get_csv_file(url = df_resource.iloc[index]["url"])
#     df_enderecamento = pd.concat(objs = [df_enderecamento, df_], ignore_index = True)\
#     .drop_duplicates()


In [20]:


# df_enderecamento = df_resources\
# .query("dataset == 'enderecamento' & format == 'CSV'")\
# .groupby("dataset")\
# .apply(lambda df: df.iloc[[-1]], include_groups = False)\
# .reset_index()\
# .apply(lambda df: get_csv_file(url = df["url"]), axis = 1)

# df_enderecamento = pd.concat(objs = list(df_enderecamento), ignore_index = True)\
# .astype(dtype = {"cep": "str", "numero_imovel": "str"})\
# .assign(numero_imovel = lambda df: df["numero_imovel"].str.replace("\\.0", "", regex = True))\
# .assign(cep = lambda df: df["cep"].str.replace("\\.0", "", regex = True))\
# .assign(urlfile = lambda df: df["urlfile"].str.split("/").apply(lambda x: x[-1]))